In [0]:
%python
from pyspark.sql.functions import col, sum as _sum, count, countDistinct, coalesce, lit, round as _round

# --- roll up costs to the encounter level ---
med_cost = (spark.read.table("meridian_dev.bronze.medications")
    .groupBy("ENCOUNTER")
    .agg(_sum("TOTALCOST").alias("med_cost")))

proc_cost = (spark.read.table("meridian_dev.bronze.procedures")
    .groupBy("ENCOUNTER")
    .agg(_sum("BASE_COST").alias("proc_cost")))

# --- conditions are diagnosed AT an encounter; attach the rolled-up cost ---
conditions = (spark.read.table("meridian_dev.bronze.conditions")
    .select(
        col("ENCOUNTER").alias("encounter_id"),
        col("CODE").alias("condition_code"),
        col("DESCRIPTION").alias("condition_desc"),
        col("PATIENT").alias("patient_id")))

cost_by_condition = (conditions
    # bring in the med + proc costs for each condition's encounter
    .join(med_cost.withColumnRenamed("ENCOUNTER", "encounter_id"), on="encounter_id", how="left")
    .join(proc_cost.withColumnRenamed("ENCOUNTER", "encounter_id"), on="encounter_id", how="left")
    # null-safe: encounters with no meds/procs contribute 0
    .withColumn("encounter_cost",
        coalesce(col("med_cost"), lit(0.0)) + coalesce(col("proc_cost"), lit(0.0)))
    # aggregate to the condition (diagnosis) grain
    .groupBy("condition_code", "condition_desc")
    .agg(
        count("*").alias("num_diagnoses"),
        countDistinct("patient_id").alias("num_patients"),
        _round(_sum("encounter_cost"), 2).alias("total_cost"),
        _round(_sum("encounter_cost") / count("*"), 2).alias("avg_cost_per_case"),
    ))

cost_by_condition.write.format("delta").mode("overwrite") \
    .saveAsTable("meridian_dev.gold.cost_by_condition")

print("Gold cost_by_condition built.")

In [0]:
SELECT condition_desc, num_patients, num_diagnoses, total_cost, avg_cost_per_case
FROM meridian_dev.gold.cost_by_condition
WHERE num_patients >= 5          -- conditions affecting a meaningful number of people
ORDER BY total_cost DESC
LIMIT 15;